# AES im Dateilabor
**M114 · Gruppenspiel · 70 Minuten**

Dieses Notebook ist euer Werkzeug für die Schlüsselstation und das Dateilabor. Ihr bedient Formulare und startet vorbereitete Zellen. **Ihr müsst keinen Code ändern.** Haltet Vermutungen, Beobachtungen und kritische Prüfungen im Word fest.

Erstellt ausschliesslich eine erfundene Testnachricht. Verwendet eine neue Übungspassphrase mit mindestens fünf unterschiedlichen Wörtern und niemals ein echtes persönliches Passwort. Die öffentliche Beispielpassphrase aus M2A ist nur ein Anschauungsbeispiel.

Es wird tatsächlich mit **AES-256-GCM** verschlüsselt. Die Bedienoberfläche und der JSON-Lerncontainer wurden für diesen Unterricht gebaut. Sie sind kein geprüftes Produkt zum Schutz echter Daten. GCM schützt den Inhalt und erkennt Veränderungen; es ist keine persönliche digitale Signatur.

**Reihenfolge:** S0 → M2A → M3A → M3B → M3C → M3D → M4A. M4B ist freiwillig. Der Spielauftrag enthält auch eine erste Mission, die ohne dieses Notebook gelöst wird.

**Dateien:** Alle Labordateien werden im linken Colab-Bereich **Dateien** unter **/content/aes_labor** bereitgestellt; die Anleitung dazu steht in M3A.


## S0 Werkzeug starten
Startet die folgende Zelle einmal und wartet auf **Werkzeug bereit**. Fehlende Pakete werden bei Bedarf automatisch installiert.

Lasst die lange Hilfszelle unverändert. Nach einem Neustart oder bei `NameError` startet ihr zuerst S0 erneut. Dadurch beginnt ein neuer Versuch im Arbeitsspeicher; vorhandene Testdateien werden nicht gelöscht. Erstellt danach die Formulare und die Testdatei erneut.


In [ ]:
#@title S0 – Werkzeug starten { display-mode: "form" }
# Eigene Unterrichtsoberfläche für erfundene Testdateien; kein Produktionsformat.
import importlib.util
import subprocess
import sys

for module_name, package_name in [
    ("cryptography", "cryptography"),
    ("ipywidgets", "ipywidgets"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("Fehlendes Unterrichtspaket wird installiert:", package_name)
        install_result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", package_name],
            capture_output=True,
            text=True,
        )
        if install_result.returncode != 0:
            raise RuntimeError(
                (
                    "Paketinstallation fehlgeschlagen. Prüft den Internetzugang und "
                    "startet S0 nochmals."
                )
            )

import base64
import hashlib
import hmac
import json
import secrets
from pathlib import Path
from copy import deepcopy
import cryptography
import ipywidgets as widgets
from IPython.display import display, clear_output
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.scrypt import Scrypt

for old_widget in globals().get("FORM_WIDGETS", []):
    old_widget.close()

MAX_TEXT_BYTES = 16 * 1024
MAX_PACKAGE_BYTES = 128 * 1024
KDF_PARAMETERS = {"name": "scrypt", "n": 2**17, "r": 8, "p": 1, "length": 32}
PUBLIC_DEMO_PASSPHRASE = "Wolke Lampe Kiesel Segel Mango"
LAB_ROOT = Path("/content") / "aes_labor"
NONCE_REGISTRY = {}
TOOL_STATE = {}
FORM_WIDGETS = []


def validate_passphrase(passphrase):
    if not isinstance(passphrase, str):
        raise ValueError("Gebt die Übungspassphrase in das Passwortfeld ein.")
    if len(passphrase.encode("utf-8")) > 1024:
        raise ValueError("Die Übungspassphrase ist für dieses Labor zu lang.")
    words = passphrase.split()
    if len(words) < 5 or len({word.casefold() for word in words}) < 5:
        raise ValueError(
            "Verwendet mindestens fünf unterschiedliche Wörter nur für diese Übung."
        )
    return passphrase.encode("utf-8")


def derive_key(passphrase, salt):
    if not isinstance(salt, bytes) or len(salt) != 16:
        raise ValueError("Der Salt muss genau 16 Bytes besitzen.")
    password_bytes = validate_passphrase(passphrase)
    return Scrypt(salt=salt, length=32, n=2**17, r=8, p=1).derive(password_bytes)


def fresh_nonce(key, random_bytes=None):
    if not isinstance(key, bytes) or len(key) != 32:
        raise ValueError("Dieses Labor erwartet einen AES-Schlüssel mit 32 Bytes.")
    random_bytes = secrets.token_bytes if random_bytes is None else random_bytes
    key_reference = hashlib.sha256(key).digest()
    used_values = NONCE_REGISTRY.setdefault(key_reference, set())
    for attempt in range(32):
        nonce = random_bytes(12)
        if not isinstance(nonce, bytes) or len(nonce) != 12:
            raise ValueError("Die Nonce muss genau 12 Bytes besitzen.")
        if nonce not in used_values:
            used_values.add(nonce)
            return nonce
    raise ValueError("Keine neue Nonce erzeugt. Startet das Werkzeug erneut.")


def encode_base64(value):
    return base64.b64encode(value).decode("ascii")


def decode_base64(value, label):
    if not isinstance(value, str):
        raise ValueError(label + " besitzt ein ungültiges Format.")
    try:
        return base64.b64decode(value, validate=True)
    except (ValueError, base64.binascii.Error):
        raise ValueError(label + " besitzt ein ungültiges Base64-Format.") from None


def canonical_header(header):
    return json.dumps(
        header, sort_keys=True, separators=(",", ":"), ensure_ascii=True
    ).encode("ascii")


def build_package(plaintext, key, salt):
    if not isinstance(plaintext, bytes) or len(plaintext) > MAX_TEXT_BYTES:
        raise ValueError("Die Testdatei darf höchstens 16 KiB enthalten.")
    if not isinstance(salt, bytes) or len(salt) != 16:
        raise ValueError("Der Salt muss genau 16 Bytes besitzen.")
    header = {
        "format": "M114-AES-GCM-LERNFORMAT",
        "version": 1,
        "algorithm": "AES-256-GCM",
        "kdf": dict(KDF_PARAMETERS),
        "salt": encode_base64(salt),
        "nonce": encode_base64(fresh_nonce(key)),
        "original_name": "original.txt",
        "content_encoding": "UTF-8",
    }
    nonce = decode_base64(header["nonce"], "Nonce")
    ciphertext_and_tag = AESGCM(key).encrypt(nonce, plaintext, canonical_header(header))
    return {"header": header, "ciphertext_and_tag": encode_base64(ciphertext_and_tag)}


def package_to_bytes(package):
    return json.dumps(package, ensure_ascii=True, sort_keys=True, indent=2).encode(
        "ascii"
    )


def unique_json_pairs(pairs):
    result = {}
    for name, value in pairs:
        if name in result:
            raise ValueError("Die Lerncontainer-Datei enthält ein doppeltes Feld.")
        result[name] = value
    return result


def parse_package(package_bytes):
    if not isinstance(package_bytes, bytes) or len(package_bytes) > MAX_PACKAGE_BYTES:
        raise ValueError("Die Lerncontainer-Datei ist zu gross oder ungültig.")
    try:
        package = json.loads(
            package_bytes.decode("utf-8"), object_pairs_hook=unique_json_pairs
        )
    except (UnicodeError, json.JSONDecodeError, RecursionError):
        raise ValueError(
            "Die Lerncontainer-Datei ist kein gültiges JSON-Dokument."
        ) from None
    if not isinstance(package, dict) or set(package) != {
        "header",
        "ciphertext_and_tag",
    }:
        raise ValueError("Die Lerncontainer-Datei besitzt unerwartete Felder.")
    header = package["header"]
    expected_fields = {
        "format",
        "version",
        "algorithm",
        "kdf",
        "salt",
        "nonce",
        "original_name",
        "content_encoding",
    }
    if not isinstance(header, dict) or set(header) != expected_fields:
        raise ValueError("Der öffentliche Header ist unvollständig oder ungültig.")
    if (
        header["format"] != "M114-AES-GCM-LERNFORMAT"
        or type(header["version"]) is not int
        or header["version"] != 1
    ):
        raise ValueError("Dieses Format wird im Unterrichtslabor nicht unterstützt.")
    if (
        header["algorithm"] != "AES-256-GCM"
        or header["original_name"] != "original.txt"
        or header["content_encoding"] != "UTF-8"
    ):
        raise ValueError("Der Header passt nicht zu diesem Unterrichtslabor.")
    parameters = header["kdf"]
    if not isinstance(parameters, dict) or set(parameters) != set(KDF_PARAMETERS):
        raise ValueError("Die Schlüsselableitungsparameter sind ungültig.")
    for name, expected_value in KDF_PARAMETERS.items():
        if (
            type(parameters[name]) is not type(expected_value)
            or parameters[name] != expected_value
        ):
            raise ValueError(
                "Die Datei enthält nicht freigegebene Schlüsselableitungsparameter."
            )
    salt = decode_base64(header["salt"], "Salt")
    nonce = decode_base64(header["nonce"], "Nonce")
    ciphertext_and_tag = decode_base64(package["ciphertext_and_tag"], "Chiffrat")
    if (
        len(salt) != 16
        or len(nonce) != 12
        or not 16 <= len(ciphertext_and_tag) <= MAX_TEXT_BYTES + 16
    ):
        raise ValueError("Salt, Nonce oder Chiffrat besitzen eine ungültige Länge.")
    return package, salt, nonce, ciphertext_and_tag


def decrypt_with_key(package_bytes, key):
    package, salt, nonce, ciphertext_and_tag = parse_package(package_bytes)
    plaintext = AESGCM(key).decrypt(
        nonce, ciphertext_and_tag, canonical_header(package["header"])
    )
    return plaintext


def decrypt_with_passphrase(package_bytes, passphrase):
    package, salt, nonce, ciphertext_and_tag = parse_package(package_bytes)
    key = derive_key(passphrase, salt)
    return AESGCM(key).decrypt(
        nonce, ciphertext_and_tag, canonical_header(package["header"])
    )


def require_current_run():
    if not TOOL_STATE.get("package_path"):
        raise ValueError(
            "Startet zuerst M3A und klickt dort auf Datei erstellen und verschlüsseln."
        )
    package_path = TOOL_STATE["package_path"]
    if not package_path.is_file() or not TOOL_STATE["original_path"].is_file():
        raise ValueError("Die Testdateien fehlen. Erstellt in M3A einen neuen Versuch.")
    return TOOL_STATE


def print_settings(state, package):
    header = package["header"]
    print("Werkzeug: Python cryptography", cryptography.__version__)
    print("Verschlüsselungsart: symmetrisch")
    print("Algorithmus und Modus:", header["algorithm"])
    print("AES-Schlüssellänge: 32 Bytes = 256 Bits")
    print("AES-Blockgrösse: 16 Bytes = 128 Bits")
    print("Schlüsselableitung: Scrypt; n = 131072, r = 8, p = 1, Ausgabe = 32 Bytes")
    print(
        (
            "Diese KDF-Parameter sind Unterrichtsparameter und keine allgemeine "
            "Produktempfehlung."
        )
    )
    print("Salt: 16 Bytes; öffentlich, nicht geheim;", header["salt"])
    print("Nonce: 12 Bytes; öffentlich, nicht geheim;", header["nonce"])
    print("GCM-Prüftag: 16 Bytes; in ciphertext_and_tag enthalten")
    print("Chiffrat und Tag (Base64-Auszug):", package["ciphertext_and_tag"][:80])
    print(
        "Base64 zeigt verschlüsselte Bytes als Zeichen; "
        "es ist keine zusätzliche Verschlüsselung."
    )
    print("Öffentlicher Header: als Zusatzdaten mit GCM geprüft")
    print("Originaldatei:", state["original_path"])
    print("Verschlüsselte Datei:", state["package_path"])
    print("Wiederhergestellte Datei:", state["recovered_path"])
    print(
        "SHA-256 Original:",
        hashlib.sha256(state["original_path"].read_bytes()).hexdigest(),
    )
    print("Originalgrösse:", state["original_path"].stat().st_size, "Bytes")
    print("JSON-Lerncontainer:", state["package_path"].stat().st_size, "Bytes")
    print("Passphrase und AES-Schlüssel werden nicht ausgegeben.")


def create_run(message, passphrase):
    if not isinstance(message, str):
        raise ValueError("Die Testnachricht muss Text sein.")
    plaintext = message.encode("utf-8")
    if len(plaintext) > MAX_TEXT_BYTES:
        raise ValueError(
            "Die Testnachricht darf höchstens 16 KiB UTF-8-Daten enthalten."
        )
    salt = secrets.token_bytes(16)
    key = derive_key(passphrase, salt)
    package = build_package(plaintext, key, salt)
    LAB_ROOT.mkdir(parents=True, exist_ok=True)
    while True:
        run_directory = LAB_ROOT / ("lauf_" + secrets.token_hex(4))
        try:
            run_directory.mkdir(exist_ok=False)
            break
        except FileExistsError:
            continue
    original_path = run_directory / "original.txt"
    package_path = run_directory / "datei.aes.json"
    recovered_path = run_directory / "wiederhergestellt.txt"
    original_path.write_bytes(plaintext)
    package_path.write_bytes(package_to_bytes(package))
    state = {
        "run_directory": run_directory,
        "original_path": original_path,
        "package_path": package_path,
        "recovered_path": recovered_path,
        "key": key,
        "salt": salt,
    }
    TOOL_STATE.clear()
    TOOL_STATE.update(state)
    return state, package


def restore_run(passphrase):
    state = require_current_run()
    package_bytes = state["package_path"].read_bytes()
    plaintext = decrypt_with_passphrase(package_bytes, passphrase)
    # AESGCM liefert Daten erst nach erfolgreicher Tag-Prüfung zurück.
    state["recovered_path"].write_bytes(plaintext)
    original = state["original_path"].read_bytes()
    return {
        "equal": hmac.compare_digest(original, plaintext),
        "original_hash": hashlib.sha256(original).hexdigest(),
        "recovered_hash": hashlib.sha256(plaintext).hexdigest(),
        "recovered_bytes": len(plaintext),
    }


def open_colab_path(path):
    path = Path(path)
    if not path.exists():
        print("Diese Datei ist noch nicht vorhanden:", path.name)
        return False
    try:
        from google.colab import files as colab_files
        colab_files.view(str(path.resolve()))
        return True
    except Exception:
        # Die Vorschau darf eine erfolgreiche Dateiverarbeitung nicht abbrechen.
        print("Öffnet links das Ordnersymbol Dateien und klickt auf Aktualisieren.")
        print("Öffnet danach:", path)
        return False


def show_colab_files(state, focus=None):
    labels = [
        ("Original anzeigen", "original_path"),
        ("Lerncontainer anzeigen", "package_path"),
        ("Wiederherstellung anzeigen", "recovered_path"),
        ("Manipulierte Kopie anzeigen", "tampered_path"),
    ]
    print("\nDateien in Colab:", state["run_directory"])
    for label, name in labels:
        path = state.get(name)
        if path is not None and Path(path).is_file():
            print("  ", Path(path).name, "–", Path(path).stat().st_size, "Bytes")
    if focus and state.get(focus) and Path(state[focus]).is_file():
        print("Jetzt bereit:", Path(state[focus]).name)
    print("Links: Dateien → Aktualisieren → aes_labor →", state["run_directory"].name)
    print("Doppelklick auf eine Datei öffnet die Vorschau in Colab.")

    folder_button = widgets.Button(
        description="Ordner in Colab öffnen",
        layout=widgets.Layout(width="290px"),
    )
    folder_button.on_click(
        lambda clicked_button, path=state["run_directory"]: open_colab_path(path)
    )
    buttons = [folder_button]
    for label, name in labels:
        path = state.get(name)
        if path is None or not Path(path).is_file():
            continue
        button = widgets.Button(
            description=label, layout=widgets.Layout(width="290px")
        )
        button.on_click(
            lambda clicked_button, file_path=path: open_colab_path(file_path)
        )
        buttons.append(button)
    FORM_WIDGETS.extend(buttons)
    display(widgets.VBox(buttons))
    # Öffnet den tatsächlich verwendeten Laufordner über Colabs Vorschau-API.
    open_colab_path(state["run_directory"])


def show_encrypt_form():
    message_input = widgets.Textarea(
        value=(
            "Dies ist unsere erfundene AES-Testdatei.\nKeine persönlichen "
            "Daten.\nGrüsse aus dem Unterricht!"
        ),
        description="Testnachricht",
        layout=widgets.Layout(width="95%", height="115px"),
        style={"description_width": "initial"},
    )
    password_input = widgets.Password(
        description="Übungspassphrase",
        placeholder="Mindestens fünf unterschiedliche Wörter",
        layout=widgets.Layout(width="95%"),
        style={"description_width": "initial"},
    )
    button = widgets.Button(
        description="Datei erstellen und verschlüsseln",
        button_style="primary",
        layout=widgets.Layout(width="320px"),
    )
    output = widgets.Output()

    def on_encrypt(clicked_button):
        passphrase = password_input.value
        password_input.value = ""
        button.disabled = True
        with output:
            clear_output(wait=True)
            try:
                state, package = create_run(message_input.value, passphrase)
                print(
                    (
                        "Verschlüsselung abgeschlossen. Die echte Testdatei und der "
                        "JSON-Lerncontainer wurden geschrieben."
                    )
                )
                print_settings(state, package)
                show_colab_files(state, focus="original_path")
            except ValueError as error:
                print("Bitte prüfen:", error)
            except (OSError, MemoryError):
                print(
                    (
                        "Versuch konnte nicht abgeschlossen werden. Prüft die "
                        "Laufzeit und "
                        "startet M3A erneut."
                    )
                )
            finally:
                passphrase = None
                button.disabled = False

    button.on_click(on_encrypt)
    form = {
        "message": message_input,
        "password": password_input,
        "button": button,
        "output": output,
    }
    FORM_WIDGETS.extend([message_input, password_input, button, output])
    display(widgets.VBox([message_input, password_input, button, output]))
    return form


def show_decrypt_form():
    password_input = widgets.Password(
        description="Übungspassphrase",
        placeholder="Dieselbe Übungspassphrase erneut eingeben",
        layout=widgets.Layout(width="95%"),
        style={"description_width": "initial"},
    )
    button = widgets.Button(
        description="Entschlüsseln und vergleichen",
        button_style="primary",
        layout=widgets.Layout(width="320px"),
    )
    output = widgets.Output()

    def on_decrypt(clicked_button):
        passphrase = password_input.value
        password_input.value = ""
        button.disabled = True
        with output:
            clear_output(wait=True)
            try:
                result = restore_run(passphrase)
                state = require_current_run()
                print(
                    "GCM-Prüfung erfolgreich. Die Datei wurde als "
                    "wiederhergestellt.txt geschrieben."
                )
                print(
                    "Bytevergleich Original / Wiederherstellung:",
                    "gleich" if result["equal"] else "verschieden",
                )
                print("SHA-256 Original:", result["original_hash"])
                print("SHA-256 Wiederherstellung:", result["recovered_hash"])
                print(
                    "SHA-256-Werte gleich:",
                    result["original_hash"] == result["recovered_hash"],
                )
                print("Wiederhergestellte Grösse:", result["recovered_bytes"], "Bytes")
                print(
                    (
                        "Der Bytevergleich prüft die Gleichheit direkt; die "
                        "Hashwerte sind "
                        "zusätzliche Belege."
                    )
                )
                show_colab_files(state, focus="recovered_path")
            except InvalidTag:
                print(
                    (
                        "GCM-Prüfung fehlgeschlagen: Passphrase oder Daten "
                        "passen nicht. "
                        "Kein Klartext freigegeben; keine Datei überschrieben."
                    )
                )
            except ValueError as error:
                print("Bitte prüfen:", error)
            except (OSError, MemoryError):
                print(
                    (
                        "Entschlüsselung konnte nicht abgeschlossen werden. Prüft die "
                        "Testdateien und die Laufzeit."
                    )
                )
            finally:
                passphrase = None
                button.disabled = False

    button.on_click(on_decrypt)
    form = {"password": password_input, "button": button, "output": output}
    FORM_WIDGETS.extend([password_input, button, output])
    display(widgets.VBox([password_input, button, output]))
    return form


def run_wrong_key_test():
    state = require_current_run()
    package_bytes = state["package_path"].read_bytes()
    original_before = state["original_path"].read_bytes()
    wrong_key = secrets.token_bytes(32)
    while hmac.compare_digest(wrong_key, state["key"]):
        wrong_key = secrets.token_bytes(32)
    try:
        decrypt_with_key(package_bytes, wrong_key)
        raise RuntimeError("Unerwartetes Testergebnis. Versuch stoppen.")
    except InvalidTag:
        print("Falscher AES-Schlüssel: GCM-Prüfung fehlgeschlagen (InvalidTag).")
        print(
            (
                "Kein Klartext freigegeben; keine Wiederherstellungsdatei "
                "geschrieben oder überschrieben."
            )
        )
    print(
        "Original unverändert:", state["original_path"].read_bytes() == original_before
    )


def run_repeat_test():
    state = require_current_run()
    plaintext = state["original_path"].read_bytes()
    first_package = build_package(plaintext, state["key"], state["salt"])
    second_package = build_package(plaintext, state["key"], state["salt"])
    first_nonce = first_package["header"]["nonce"]
    second_nonce = second_package["header"]["nonce"]
    print("Nachricht gleich: True; AES-Schlüssel gleich: True; Salt gleich: True")
    print("Nonce Versuch 1:", first_nonce)
    print("Nonce Versuch 2:", second_nonce)
    print("Nonce verschieden:", first_nonce != second_nonce)
    print(
        "Chiffrat samt Tag verschieden:",
        first_package["ciphertext_and_tag"] != second_package["ciphertext_and_tag"],
    )
    print("Hier wurde nur die Nonce geändert. Salt und Schlüssel blieben gleich.")
    print("Die Originaldateien des Arbeitsversuchs wurden nicht verändert.")
    return first_package, second_package


def run_tamper_test():
    state = require_current_run()
    original_package_bytes = state["package_path"].read_bytes()
    package, salt, nonce, ciphertext_and_tag = parse_package(original_package_bytes)
    if len(ciphertext_and_tag) <= 16:
        raise ValueError(
            (
                "Für den Manipulationstest braucht ihr mindestens ein Zeichen "
                "Testnachricht. Erstellt in M3A einen neuen Versuch."
            )
        )
        # Kontrollentschlüsselung verändert oder verbraucht die Nonce nicht.
    control_plaintext = decrypt_with_key(original_package_bytes, state["key"])
    changed_data = bytearray(ciphertext_and_tag)
    changed_data[0] ^= 1
    tampered_package = deepcopy(package)
    tampered_package["ciphertext_and_tag"] = encode_base64(bytes(changed_data))
    tampered_bytes = package_to_bytes(tampered_package)
    tampered_path = state["run_directory"] / "manipulierte_kopie.aes.json"
    tampered_path.write_bytes(tampered_bytes)
    print("Kontrolle Originalpaket: GCM-Prüfung erfolgreich.")
    print(
        "Originalkontrolle entspricht original.txt:",
        control_plaintext == state["original_path"].read_bytes(),
    )
    try:
        decrypt_with_key(tampered_bytes, state["key"])
        raise RuntimeError("Unerwartetes Testergebnis. Versuch stoppen.")
    except InvalidTag:
        print("Manipulierte Kopie: GCM-Prüfung fehlgeschlagen (InvalidTag).")
        print(
            (
                "Kein veränderter Klartext freigegeben; wiederhergestellt.txt wird "
                "nicht überschrieben."
            )
        )
    print(
        "Originalpaket unverändert:",
        state["package_path"].read_bytes() == original_package_bytes,
    )
    print("Manipulierte Kopie:", tampered_path)
    TOOL_STATE["tampered_path"] = tampered_path
    show_colab_files(state, focus="tampered_path")
    return tampered_bytes


def run_hash_comparison():
    state = require_current_run()
    tampered_path = state.get("tampered_path")
    if tampered_path is None or not tampered_path.is_file():
        raise ValueError("Startet zuerst M4A, damit die manipulierte Kopie existiert.")
    original_hash = hashlib.sha256(state["package_path"].read_bytes()).hexdigest()
    tampered_hash = hashlib.sha256(tampered_path.read_bytes()).hexdigest()
    print("Öffentlicher SHA-256 des Originalpakets:", original_hash)
    print(
        "Neu berechneter öffentlicher SHA-256 der manipulierten Kopie:", tampered_hash
    )
    print("Die Werte sind verschieden:", original_hash != tampered_hash)
    print(
        "Jede Person kann den öffentlichen Hash einer veränderten Datei neu berechnen."
    )
    print(
        (
            "Ohne eine unabhängig vertrauenswürdige Referenz beweist dieser Hash "
            "keine Echtheit."
        )
    )
    print("Für einen passenden neuen GCM-Tag benötigt man den geheimen AES-Schlüssel.")
    print("Ein ungeschützter öffentlicher Hash ersetzt die GCM-Prüfung deshalb nicht.")


print("Werkzeug bereit. Öffnet danach M2A.")
print("Alle Labordateien werden im Colab-Bereich Dateien unter /content/aes_labor angelegt.")
print(
    (
        "Lernformat für erfundene Unterrichtsdateien. Keine echten "
        "Passwörter oder vertraulichen Daten verwenden."
    )
)
print(
    (
        "Beim erneuten Start von S0 werden der Arbeitsspeicher und die "
        "Formulare zurückgesetzt. Vorhandene Testdateien bleiben erhalten."
    )
)

## M2A Schlüsselstation – Passphrase und Salt

**Super einfach erklärt**

- **Passphrase:** Die Passphrase ist eure geheime Folge aus Wörtern, aus der das Werkzeug den AES-Schlüssel erzeugt.
- **Salt:** Der Salt ist eine zufällige öffentliche Zugabe, damit aus derselben Passphrase nicht immer derselbe Schlüssel entsteht.

**Vermutet zuerst:** Entstehen bei derselben Passphrase und demselben Salt gleiche Schlüssel, und was ändert ein anderer Salt?

Startet die folgende Zelle; ihr müsst nichts eingeben und keinen Code verändern.
Sie verwendet die öffentliche Übungspassphrase **Wolke Lampe Kiesel Segel Mango** und erzeugt zwei neue Salts.
Die geheimen Schlüssel werden nicht ausgegeben.

Notiert die Schlüssellänge und die beiden Vergleiche: gleicher Salt sowie anderer Salt.
Die konkreten Saltwerte können bei jedem Start anders aussehen.
Das öffentliche Beispiel darf keine echten Daten schützen.


In [ ]:
#@title M2A – Schlüsselstation { display-mode: "form" }
try:
    first_salt = secrets.token_bytes(16)
    second_salt = secrets.token_bytes(16)
    while second_salt == first_salt:
        second_salt = secrets.token_bytes(16)
    first_key = derive_key(PUBLIC_DEMO_PASSPHRASE, first_salt)
    repeated_key = derive_key(PUBLIC_DEMO_PASSPHRASE, first_salt)
    second_key = derive_key(PUBLIC_DEMO_PASSPHRASE, second_salt)
    print("Öffentliche Übungspassphrase:", PUBLIC_DEMO_PASSPHRASE)
    print("Salt 1:", encode_base64(first_salt), "(16 Bytes; nicht geheim)")
    print("Salt 2:", encode_base64(second_salt), "(16 Bytes; nicht geheim)")
    print("AES-Schlüssellänge:", len(first_key), "Bytes =", len(first_key) * 8, "Bits")
    print(
        "Gleiche Passphrase + gleicher Salt → gleicher Schlüssel:",
        hmac.compare_digest(first_key, repeated_key),
    )
    print(
        "Gleiche Passphrase + anderer Salt → anderer Schlüssel:",
        not hmac.compare_digest(first_key, second_key),
    )
    print(
        "Scrypt verlangt Rechenarbeit. Die Passphrase ist nicht "
        "selbst der AES-Schlüssel."
    )
    print("Die Schlüsselbytes werden nicht angezeigt.")
except NameError:
    print("Startet zuerst S0.")

## M3A Eine eigene Testdatei erstellen und verschlüsseln

1. Startet die Zelle; das Formular erscheint.
2. Ergänzt im Feld **Testnachricht** eine eigene erfundene Zeile.
3. Gebt im Feld **Übungspassphrase** mindestens fünf unterschiedliche Wörter nur für diesen Versuch ein und merkt sie euch.
4. Klickt **Datei erstellen und verschlüsseln**; danach wird das Passphrasenfeld geleert.
5. **original.txt** und **datei.aes.json** liegen jetzt im Colab-Dateienbereich im angezeigten Laufordner.
6. Übertragt die Einstellungen und Dateigrössen ins Arbeitsheft, aber niemals die Passphrase oder den AES-Schlüssel.

### So findet ihr die Dateien in Colab

Klickt auf **Ordner in Colab öffnen**, um den aktuellen Laufordner zu öffnen.
Die Schaltflächen **Original anzeigen** und **Lerncontainer anzeigen** öffnen eine Vorschau direkt in Colab.

Ihr könnt den Ordner auch selbst öffnen:

1. Klickt links auf das **Ordnersymbol Dateien**.
2. Klickt oben im Dateienbereich auf **Aktualisieren** – den runden Pfeil.
3. Öffnet **aes_labor** und darin den angezeigten Ordner **lauf_…**.
4. Öffnet die gewünschte Datei mit einem **Doppelklick**.

Der feste Hauptordner ist **/content/aes_labor**; für jeden Versuch entsteht darin ein neuer Laufordner.
Die Labordateien werden direkt dort gespeichert und bleiben in der aktuellen Colab-Laufzeit.

### Die Angaben in der Ausgabe – je ein einfacher Satz

| Begriff | Super einfache Erklärung |
|---|---|
| Symmetrisch | Zum Verschlüsseln und Entschlüsseln braucht ihr denselben geheimen Schlüssel. |
| AES-256-GCM | AES-256-GCM verschlüsselt eure Datei mit einem 256-Bit-Schlüssel und prüft die geschützten Daten auf Veränderungen. |
| AES-Schlüssellänge | Der geheime AES-Schlüssel ist hier 256 Bits lang, das sind 32 Bytes. |
| AES-Blockgrösse | AES verarbeitet Blöcke von 128 Bits, also 16 Bytes. |
| Schlüsselableitung | Die Schlüsselableitung macht aus eurer Passphrase und dem Salt den passenden AES-Schlüssel. |
| n bei scrypt | n bestimmt den grundlegenden Arbeitsaufwand der Schlüsselableitung. |
| r bei scrypt | r beeinflusst den Speicherbedarf und die Rechenarbeit der Schlüsselableitung. |
| p bei scrypt | p bestimmt, wie viele unabhängige Rechenläufe scrypt ausführt. |
| Nonce | Die Nonce ist ein öffentlicher Einmalwert, der bei jeder neuen Verschlüsselung mit demselben Schlüssel neu sein muss. |
| GCM-Prüfung | Die GCM-Prüfung gibt Daten nur frei, wenn Schlüssel, Nonce und geschützte Daten zur Prüfmarke passen. |

Die Werte **n, r und p** sind vorbereitet; ihr müsst sie weder berechnen noch ändern.
Die Blockgrösse bedeutet nicht, dass eure Datei genau 16 Bytes gross sein muss.


In [ ]:
#@title M3A – Formular zum Verschlüsseln { display-mode: "form" }
try:
    encrypt_form = show_encrypt_form()
except NameError:
    print("Startet zuerst S0.")

## M3B Entschlüsseln und vergleichen

**SHA-256 in einem Satz:** SHA-256 berechnet einen digitalen Fingerabdruck, mit dem ihr zwei Dateien vergleichen könnt.

1. Startet die Zelle und gebt zuerst absichtlich eine **andere Übungspassphrase aus mindestens fünf unterschiedlichen Wörtern** ein.
2. Klickt **Entschlüsseln und vergleichen** und notiert die Meldung.
3. Gebt danach die **richtige Übungspassphrase aus M3A** ein und klickt nochmals; Gross- und Kleinschreibung sowie Leerzeichen müssen stimmen.
4. Erst wenn die GCM-Prüfung erfolgreich ist, wird **wiederhergestellt.txt** im **selben Colab-Laufordner** wie Original und Lerncontainer gespeichert.
5. Klickt **Wiederherstellung anzeigen**, um die Datei direkt in Colab zu lesen, oder **Ordner in Colab öffnen**.
6. Fehlt die Datei in der linken Liste noch, klickt dort auf **Aktualisieren** und öffnet **aes_labor → lauf_… → wiederhergestellt.txt**.

Vergleicht Text, Bytevergleich und die beiden vollständigen SHA-256-Werte.
Notiert im Arbeitsheft, ob die Dateien gleich sind.
Der Bytevergleich prüft die Gleichheit direkt; SHA-256 ist ein zusätzlicher Vergleich und keine Verschlüsselung.
Ein Fehler allein sagt nicht sicher, ob die Passphrase falsch oder die Datei verändert ist.
Nach jedem Klick wird das Passphrasenfeld geleert.


In [ ]:
#@title M3B – Formular zum Entschlüsseln { display-mode: "form" }
try:
    decrypt_form = show_decrypt_form()
except NameError:
    print("Startet zuerst S0.")

## M3C Einen falschen AES-Schlüssel testen
**Vermutet zuerst:** Gibt das Werkzeug einen halb lesbaren Text aus oder verweigert es die Freigabe?

Die Zelle erzeugt einen zufälligen falschen 256-Bit-Schlüssel. Sie versucht damit, denselben Lerncontainer zu entschlüsseln. Ihr müsst nichts eingeben. Eine fehlgeschlagene GCM-Prüfung liefert keinen Klartext. Vorhandene Dateien werden dabei nicht überschrieben.

Notiert die Beobachtung. Ein `InvalidTag` sagt aus, dass Schlüssel und geprüfte Daten nicht zusammenpassen; es beweist allein nicht, welche Ursache vorliegt.


In [ ]:
#@title M3C – falscher Schlüssel { display-mode: "form" }
try:
    run_wrong_key_test()
except NameError:
    print("Startet zuerst S0 und erstellt in M3A eine Testdatei.")
except ValueError as error:
    print("Bitte prüfen:", error)

## M3D Dieselbe Nachricht zweimal verschlüsseln

**Nonce Versuch in einem Satz:** «Nonce Versuch 1» und «Nonce Versuch 2» sind die zwei frischen Einmalwerte für die zwei Verschlüsselungen derselben Nachricht.

**Vermutet zuerst:** Müssen die verschlüsselten Daten gleich sein, wenn Nachricht und Schlüssel gleich bleiben?

Startet die Zelle; ihr müsst nichts eingeben.
**Nachricht, AES-Schlüssel und Salt bleiben gleich**, nur die Nonce wird für jede der beiden Verschlüsselungen frisch erzeugt.
Die Originaldateien aus M3A bleiben unverändert.

Notiert, ob die beiden Nonces gleich oder verschieden sind und was ihr beim Chiffrat beobachtet.
Mit demselben Schlüssel darf eine Nonce nie erneut **zum Verschlüsseln** verwendet werden.
Mehrfaches Entschlüsseln eines bereits vorhandenen Pakets ist dagegen erlaubt.


In [ ]:
#@title M3D – kontrollierter Vergleich mit zwei frischen Nonces { display-mode: "form" }
try:
    repeated_packages = run_repeat_test()
except NameError:
    print("Startet zuerst S0 und erstellt in M3A eine Testdatei.")
except ValueError as error:
    print("Bitte prüfen:", error)

## M4A Manipulation an einer Kopie erkennen

**Vermutet zuerst:** Reicht ein verändertes Byte, um die Freigabe zu verhindern?

1. Startet die Zelle; sie prüft zuerst den unveränderten Lerncontainer mit dem richtigen Schlüssel.
2. Danach verändert sie genau ein Chiffratbyte einer **Kopie**.
3. Diese Kopie wird als **manipulierte_kopie.aes.json** im **aktuellen Colab-Laufordner** gespeichert.
4. Klickt **Manipulierte Kopie anzeigen**, um sie direkt in Colab zu öffnen.
5. Alternativ: links **Dateien → Aktualisieren → aes_labor → angezeigter lauf_…-Ordner → manipulierte_kopie.aes.json**.

Ihr müsst keinen alten Laufordner aus einer früheren Ausgabe suchen.
Der aktuelle Pfad und alle tatsächlich vorhandenen Dateien werden nach dem Versuch neu angezeigt.
Vergleicht die erfolgreiche Kontrolle mit der fehlgeschlagenen Prüfung der Kopie und notiert eure Erklärung.

Der ursprüngliche Lerncontainer, original.txt und eine bereits gültige Wiederherstellung bleiben unverändert.
Die Testnachricht muss für diesen Versuch mindestens ein Zeichen enthalten.


In [ ]:
#@title M4A – Byte in einer Kopie verändern { display-mode: "form" }
try:
    tampered_package_bytes = run_tamper_test()
except NameError:
    print("Startet zuerst S0 und erstellt in M3A eine Testdatei.")
except ValueError as error:
    print("Bitte prüfen:", error)

## M4B Freiwillig – warum ein öffentlicher Hash nicht genügt

**Super einfach erklärt**

- **Öffentlicher Hash:** Ein öffentlicher Hash ist ein Dateifingerabdruck, den jede Person ohne geheimen Schlüssel berechnen kann.
- **GCM:** GCM ergänzt AES um eine schlüsselabhängige Prüfung, die Veränderungen an den geschützten Daten erkennt.

Startet zuerst M4A und danach diese Zelle.
Sie berechnet einen neuen SHA-256-Wert für die bereits in Colab gespeicherte manipulierte Kopie.
Ihr müsst nichts eingeben.

Notiert: Ein neuer öffentlicher Hash lässt sich auch für eine geänderte Datei berechnen.
Einen dazu passenden neuen GCM-Tag kann eine Person ohne den geheimen AES-Schlüssel nicht einfach ebenso erzeugen.
Ein GCM-Tag beweist nicht, welche Person die Datei erstellt hat.


In [ ]:
#@title M4B – öffentlicher Hash und GCM-Prüfung { display-mode: "form" }
try:
    run_hash_comparison()
except NameError:
    print("Startet zuerst S0.")
except ValueError as error:
    print("Bitte prüfen:", error)

## Abschluss
Prüft eure Dokumentation: Enthält jeder Befund eine Vermutung, eine konkrete Beobachtung und eine Erklärung? Notiert keine Passphrase und keine geheimen Schlüssel.

Gebt die Dokumentation gemäss Spielauftrag ab. Notebook und Labordateien sind Arbeitswerkzeuge. Die Dateien dürfen weiter als erfundene Unterrichtsbeispiele genutzt werden; für echte Daten braucht es ein geeignetes geprüftes Werkzeug und eine korrekte Anwendung.


Copyright 2026 [Markus Ineichen](mailto:markus.ineichen1@sluz.ch)

Code licence: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)